# Phase 2 — CNN Embedding Evaluation Battery (18 Tests)
## nnUNet v2 PlainConvUNet — Baseline for ViT Comparison

**Model**: nnUNet PlainConvUNet | 30.8M params | 6-stage encoder  
**Training**: 28 epochs on BraTS 2024 (Post-Treatment Glioma)  
**Best Dice**: 0.8150 (epoch 27) → WT=0.864, TC=0.787, ET=0.794  
**Embedding**: 2825-D (octant=2048 + region=768 + vol=9)

**Tests:**
- **M1-M6**: Morphology (volume, necrosis, core fraction, patient purity)
- **H1-H5**: Heterogeneity (RankMe, Diversity, Uniformity, Responder F1, Norm CV)
- **T1-T8**: Temporal (Spearman ρ, ordering, RANO AUC, coherence, Kendall τ)

**Purpose:** Establish baseline performance → prove ViT (Phase 3) improves on CNN limitations.


In [ ]:
# ═══════════════════════════════════════════════════════════
# CACHE MODE: if cnn_eval_results.json already exists,
# load it and skip recomputing all 18 tests.
# Set SKIP_IF_CACHED = False to force recompute.
# ═══════════════════════════════════════════════════════════
SKIP_IF_CACHED = False

import numpy as np, json as _json, random, warnings, os
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.manifold import TSNE
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.metrics import r2_score, f1_score, roc_auc_score
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from scipy.stats import pearsonr, spearmanr, kendalltau as kt
import pandas as pd
warnings.filterwarnings("ignore")
np.random.seed(42)

OUTPUT_ROOT = Path("/kaggle/working/phase2_evaluation")
FIG_DIR = OUTPUT_ROOT / "figures"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

SEARCH_ROOTS = [Path("/kaggle/input"), Path("/kaggle/working")]

def find_npz(patterns):
    for pat in patterns:
        for root in SEARCH_ROOTS:
            matches = list(root.rglob(f"{pat}.npz"))
            if matches:
                return matches[0]
    return None

def find_file(patterns):
    """Find any file matching name patterns across search roots."""
    for pat in patterns:
        for root in SEARCH_ROOTS:
            matches = list(root.rglob(pat))
            if matches:
                return matches[0]
    return None

def load_embedding(path, model_key):
    data = np.load(path, allow_pickle=True)
    embs_arr = data["embeddings"]     # (N, 2825)
    pids_arr = data["patient_ids"]    # (N,)
    tps_arr  = data["timepoints"]     # (N,)
    emb_dict = {}
    for i in range(len(embs_arr)):
        key = f"{pids_arr[i]}__{tps_arr[i]}"
        emb_dict[key] = embs_arr[i]
    return emb_dict, embs_arr.shape[1]

def match_row(tumor_df, pid, tp):
    if tumor_df is None: return None
    m = tumor_df[(tumor_df["patient_id"].astype(str) == str(pid)) &
                 (tumor_df["timepoint"].astype(str) == str(tp))]
    if len(m) > 0: return m.iloc[0]
    tp_int = int(tp) - 100 if str(tp).isdigit() and int(tp) >= 100 else int(tp)
    m = tumor_df[(tumor_df["patient_id"].astype(str) == str(pid)) &
                 (tumor_df["timepoint"].astype(str) == str(tp_int))]
    if len(m) > 0: return m.iloc[0]
    return None

# ── Try loading cached results ──
cached_json = find_file(["cnn_eval_results.json"])
CACHE_LOADED = False

if SKIP_IF_CACHED and cached_json is not None:
    with open(cached_json) as f:
        results = _json.load(f)
    # Copy results to output so plots can use them
    out_path = OUTPUT_ROOT / "cnn_eval_results.json"
    if str(cached_json) != str(out_path):
        with open(out_path, "w") as f:
            _json.dump(results, f, indent=2)
    models = {k: {} for k in results}  # empty dicts — only needed for model name list
    CACHE_LOADED = True
    print(f"✅ CACHE LOADED: {cached_json}")
    print(f"   Models: {list(results.keys())}")
    print(f"   Metrics: {len(list(results.values())[0])} per model")
    print("   → Skipping tests M1-T8. Running plots + dashboard only.")
else:
    # ── Load CNN embeddings ──
    cnn_path = find_npz(["cnn_nnunet_embeddings", "nnunet_embeddings"])
    models = {}
    if cnn_path:
        embs_cnn, dim_cnn = load_embedding(cnn_path, "nnunet")
        models["nnunet"] = embs_cnn
        print(f"nnunet: {len(embs_cnn)} scans | dim={dim_cnn}")
        print(f"  File: {cnn_path}")
    else:
        raise FileNotFoundError(
            "No CNN embeddings found. Attach cnn_nnunet_embeddings.npz as input.\n"
            "Or attach cnn_eval_results.json to skip recomputation.")

    # ── RAW EMBEDDING STATISTICS ──
    print("\n" + "="*60)
    print("  RAW EMBEDDING STATISTICS")
    print("="*60)
    for mn, emb_dict in models.items():
        keys = sorted(emb_dict.keys())
        X_raw = np.stack([emb_dict[k] for k in keys])
        norms = np.linalg.norm(X_raw, axis=1)
        pids = set(k.split("__")[0] for k in keys)
        D = X_raw.shape[1]
        print(f"\n  {mn}: {len(keys)} scans | {len(pids)} patients | dim={D}")
        print(f"    Norm: min={norms.min():.1f}  max={norms.max():.1f}  mean={norms.mean():.1f}")
        print(f"    Norm CV (std/mean): {norms.std()/norms.mean():.4f}")
        if D >= 2121:
            C = (D - 9) // 11
            print(f"    Architecture: C={C} | octant={8*C}D  region={3*C}D  vol=9D")
        idx = np.random.choice(len(keys), size=min(1000, len(keys)), replace=False)
        pairs = [(idx[i], idx[i+1]) for i in range(0, len(idx)-1, 2)]
        cos_sims = []
        for a, b in pairs[:500]:
            na, nb_ = np.linalg.norm(X_raw[a]), np.linalg.norm(X_raw[b])
            if na > 1e-8 and nb_ > 1e-8:
                cos_sims.append(np.dot(X_raw[a], X_raw[b]) / (na * nb_))
        cos_sims = np.array(cos_sims)
        print(f"    Cosine sim ({len(cos_sims)} pairs): mean={cos_sims.mean():.3f}  std={cos_sims.std():.3f}")
        X_unit = X_raw / (norms[:, None] + 1e-8)
        subset = X_unit[np.random.choice(len(X_unit), min(500, len(X_unit)), replace=False)]
        dists = [np.linalg.norm(subset[i] - subset[j])
                 for i in range(len(subset)) for j in range(i+1, min(i+20, len(subset)))]
        print(f"    Diversity (mean pairwise L2): {np.mean(dists):.3f}")
        svs = np.linalg.svd(X_unit[:min(500, len(X_unit))], compute_uv=False)
        p = svs / svs.sum(); p = p[p > 1e-10]
        print(f"    RankMe (effective rank): {float(np.exp(-np.sum(p * np.log(p)))):.1f}")
        print(f"    Zero embeddings (resected): {np.sum(norms < 1e-5)}")

    # ── COMPONENT-WISE L2 NORMALISATION ──
    print("\n" + "="*60)
    print("  APPLYING COMPONENT-WISE L2 NORMALISATION")
    print("="*60)
    for mn in list(models.keys()):
        emb_dict = models[mn]
        keys = list(emb_dict.keys())
        arr = np.stack([emb_dict[k] for k in keys])
        D = arr.shape[1]
        if D >= 2121:
            C = (D - 9) // 11
            oct_d, reg_d = 8 * C, 3 * C
            comp_o = arr[:, :oct_d] / (np.linalg.norm(arr[:, :oct_d], axis=1, keepdims=True) + 1e-8)
            comp_r = arr[:, oct_d:oct_d+reg_d] / (np.linalg.norm(arr[:, oct_d:oct_d+reg_d], axis=1, keepdims=True) + 1e-8)
            comp_v = arr[:, oct_d+reg_d:] / (np.linalg.norm(arr[:, oct_d+reg_d:], axis=1, keepdims=True) + 1e-8)
            arr_n = np.concatenate([comp_o, comp_r, comp_v * 2], axis=1)
            print(f"  {mn}: Component-wise L2 (D={D})")
        else:
            arr_n = arr / (np.linalg.norm(arr, axis=1, keepdims=True) + 1e-8)
            print(f"  {mn}: Global L2 (D={D})")
        models[mn] = {k: arr_n[i] for i, k in enumerate(keys)}
        norms_n = np.linalg.norm(arr_n, axis=1)
        idx = np.random.choice(len(keys), min(1000, len(keys)), replace=False)
        pairs = [(idx[i], idx[i+1]) for i in range(0, len(idx)-1, 2)]
        cos_p = [np.dot(arr_n[a], arr_n[b]) / (np.linalg.norm(arr_n[a])*np.linalg.norm(arr_n[b]) + 1e-8)
                 for a, b in pairs[:500]
                 if np.linalg.norm(arr_n[a]) > 1e-8 and np.linalg.norm(arr_n[b]) > 1e-8]
        print(f"    Full ({D}-D): mean_norm={norms_n.mean():.3f}  CV={norms_n.std()/norms_n.mean():.4f}")
        print(f"    Cosine sim ({len(cos_p)} pairs): mean={np.mean(cos_p):.3f}  std={np.std(cos_p):.3f}")
        X_u = arr_n / (norms_n[:, None] + 1e-8)
        sub = X_u[np.random.choice(len(X_u), min(500, len(X_u)), replace=False)]
        dd = [np.linalg.norm(sub[i]-sub[j]) for i in range(len(sub)) for j in range(i+1, min(i+20, len(sub)))]
        print(f"    Diversity: {np.mean(dd):.3f}")

    # ── LOAD TUMOUR VOLUME METADATA ──
    tumor_df = None
    for root in SEARCH_ROOTS:
        for f in root.rglob("tumor_volumes.csv"):
            tumor_df = pd.read_csv(f)
            print(f"\nTumor volumes: {len(tumor_df)} rows from {f}")
            break
        if tumor_df is not None: break
    if tumor_df is not None:
        keys = list(list(models.values())[0].keys())
        pid0, tp0 = keys[0].split("__")
        r = match_row(tumor_df, pid0, tp0)
        print(f"Match test: {'FOUND' if r is not None else 'NOT FOUND'}")

    results = {}

print(f"\n✅ Ready | CACHE_LOADED={CACHE_LOADED} | Models: {list(models.keys())}")
print(f"Output: {OUTPUT_ROOT}")


In [ ]:
if not CACHE_LOADED:
    # M1-M6: MORPHOLOGY TESTS (REVISED — nonlinear probes + L2 norm)
    print("=" * 60)
    print("  MORPHOLOGY TESTS M1-M6")
    print("=" * 60)
    
    for mn, embs in models.items():
        keys = list(embs.keys())
        X    = np.stack([embs[k] for k in keys])
        # ── L2 normalise (recommendation: fix H4 norm variance) ──
        X_l2 = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-8)
        Xs   = StandardScaler().fit_transform(X_l2)  # scale L2-normed
        if mn not in results: results[mn] = {}
    
        mX, mvols = [], {"wt": [], "tc": [], "et": []}
        for k in keys:
            pid, tp = k.split("__")
            row = match_row(tumor_df, pid, tp)
            if row is not None:
                mX.append(Xs[keys.index(k)])
                for r_name in ["wt", "tc", "et"]:
                    for col in [f"{r_name}_vol", f"{r_name.upper()}_vol",
                                 f"{r_name}_volume", f"vol_{r_name}"]:
                        if col in row.index:
                            mvols[r_name].append(float(row[col])); break
                    else:
                        mvols[r_name].append(0.0)
    
        print("{}: {}/{} matched".format(mn, len(mX), len(keys)))
        if len(mX) < 10:
            print("  Too few matched — skipping regression tests")
            # M6 still runs (label-free)
            pids_arr = np.array([k.split("__")[0] for k in keys])
            nbrs = NearestNeighbors(n_neighbors=11).fit(Xs)
            _, idx = nbrs.kneighbors(Xs)
            results[mn]["M6_patient_purity_pct"] = float(
                100 * np.mean([np.mean(pids_arr[idx[i,1:]] == pids_arr[i])
                               for i in range(len(keys))]))
            continue
    
        mX = np.stack(mX)
        y_wt = np.array(mvols["wt"])
        y_tc = np.array(mvols["tc"])
        y_et = np.array(mvols["et"])
        ridge = Ridge(alpha=1.0)
        rf    = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    
        # ── M1: Volume R² — Ridge (linear) + RF (nonlinear) + Spearman ──
        p_ridge = cross_val_predict(ridge, mX, y_wt, cv=5)
        p_rf    = cross_val_predict(rf,    mX, y_wt, cv=5)
        results[mn]["M1_volume_R2_ridge"]  = float(r2_score(y_wt, p_ridge))  # expected negative
        results[mn]["M1_volume_R2_rf"]     = float(r2_score(y_wt, p_rf))     # nonlinear probe
        results[mn]["M1_spearman_rho"]     = float(spearmanr(p_ridge, y_wt)[0])
    
        # ── M2: Log-Volume R² — Ridge + RF ──
        yl = np.log1p(y_wt)
        results[mn]["M2_logvol_R2_ridge"] = float(r2_score(yl, cross_val_predict(ridge, mX, yl, cv=5)))
        results[mn]["M2_logvol_R2_rf"]    = float(r2_score(yl, cross_val_predict(rf,    mX, yl, cv=5)))
    
        # ── M3: Enhancement Fraction — Ridge + RF ──
        y_ef = y_et / (y_wt + 0.01)
        results[mn]["M3_enhancement_ridge"] = float(r2_score(y_ef, cross_val_predict(ridge, mX, y_ef, cv=5)))
        results[mn]["M3_enhancement_rf"]    = float(r2_score(y_ef, cross_val_predict(rf,    mX, y_ef, cv=5)))
    
        # ── M4: Necrosis F1 (unchanged) ──
        ncr   = y_tc - y_et
        y_ncr = ((ncr / (y_tc + 1e-6)) > 0.10).astype(int)
        if len(set(y_ncr)) >= 2:
            p_ncr = cross_val_predict(LogisticRegression(max_iter=1000), mX, y_ncr, cv=5)
            results[mn]["M4_necrosis_F1"] = float(f1_score(y_ncr, p_ncr, average="weighted"))
        else:
            results[mn]["M4_necrosis_F1"] = 0.0
    
        # ── M5: Core Fraction — Ridge + RF ──
        y_cf = y_tc / (y_wt + 0.01)
        results[mn]["M5_corefrac_ridge"] = float(r2_score(y_cf, cross_val_predict(ridge, mX, y_cf, cv=5)))
        results[mn]["M5_corefrac_rf"]    = float(r2_score(y_cf, cross_val_predict(rf,    mX, y_cf, cv=5)))
    
        # ── M6: Patient Identity Purity (label-free, on L2-normed space) ──
        pids_arr = np.array([k.split("__")[0] for k in keys])
        nbrs = NearestNeighbors(n_neighbors=11).fit(Xs)
        _, idx = nbrs.kneighbors(Xs)
        results[mn]["M6_patient_purity_pct"] = float(
            100 * np.mean([np.mean(pids_arr[idx[i,1:]] == pids_arr[i])
                           for i in range(len(keys))]))
    
        for k, v in sorted(results[mn].items()):
            if k.startswith("M"):
                print("  {} {}: {:.3f}".format(mn, k, v))
    

In [ ]:
if not CACHE_LOADED:
    # H1-H5: HETEROGENEITY TESTS (REVISED)
    # H1: RankMe + effective rank (replaces PCA range)
    # H2: 2000 pairs + Uniformity (Wang & Isola 2020)
    # H3: Treatment responder F1 (replaces IDH)
    # H5: RankMe standalone (new)
    print('\n' + '='*60)
    print('  HETEROGENEITY TESTS H1-H5')
    print('='*60)
    
    for mn, embs in models.items():
        keys = list(embs.keys())
        X    = np.stack([embs[k] for k in keys])
        # L2-normalise before all heterogeneity metrics
        X_l2 = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-8)
        Xs   = StandardScaler().fit_transform(X_l2)
    
        # H1 — RankMe + Effective Rank (replaces PCA cumulative variance)
        U, S, Vh = np.linalg.svd(Xs, full_matrices=False)
        p_sv = S / S.sum()
        rankme = float(np.exp(-np.sum(p_sv * np.log(p_sv + 1e-12))))
        eff_rank_95 = int(np.searchsorted(np.cumsum(p_sv), 0.95)) + 1
        results[mn]['H1_rankme']       = rankme
        results[mn]['H1_eff_rank_95']  = float(eff_rank_95)
    
        # H2 — Diversity (2000 pairs) + Uniformity (Wang & Isola 2020)
        emb_n = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-8)
        idx2  = np.random.choice(len(keys), (2000, 2), replace=True)
        sims  = (emb_n[idx2[:,0]] * emb_n[idx2[:,1]]).sum(1)
        results[mn]['H2_diversity'] = float(1 - np.mean(sims))
        sq_dist = np.sum((emb_n[idx2[:,0]] - emb_n[idx2[:,1]])**2, axis=1)
        results[mn]['H2_uniformity'] = float(np.log(np.mean(np.exp(-2 * sq_dist))))
    
        # H3 — Treatment Responder F1 (REPLACES IDH)
        # responder if WT volume decreases > 20% from first to last visit
        pe = {}
        for k in keys:
            pid, tp = k.split('__')
            if pid not in pe: pe[pid] = {}
            pe[pid][tp] = embs[k]
        X_resp, y_resp = [], []
        if tumor_df is not None:
            for pid, tps in pe.items():
                if len(tps) < 2: continue
                stps = sorted(tps.keys())
                v0 = tumor_df[(tumor_df['patient_id']==pid) & (tumor_df['timepoint']==int(stps[0]))]
                vT = tumor_df[(tumor_df['patient_id']==pid) & (tumor_df['timepoint']==int(stps[-1]))]
                if len(v0) > 0 and len(vT) > 0:
                    ratio = vT.iloc[0]['wt_vol'] / (v0.iloc[0]['wt_vol'] + 1e-6)
                    y_resp.append(1 if ratio < 0.80 else 0)  # responder: -20%
                    X_resp.append(Xs[keys.index(f'{pid}__{stps[0]}')])
        if len(X_resp) >= 10 and len(set(y_resp)) >= 2:
            scores = cross_val_score(LogisticRegression(max_iter=1000),
                                     np.stack(X_resp), y_resp, cv=5, scoring='f1_weighted')
            results[mn]['H3_responder_F1'] = float(scores.mean())
        else:
            results[mn]['H3_responder_F1'] = 0.0
    
        # H4 — Norm CV (raw embeddings)
        norms = np.linalg.norm(X, axis=1)
        results[mn]['H4_norm_cv'] = float(np.std(norms) / (np.mean(norms) + 1e-8))
    
        # H5 — RankMe standalone (new test)
        results[mn]['H5_rankme_standalone'] = rankme
    
        for k, v in sorted(results[mn].items()):
            if k.startswith('H'):
                print(f'  {mn} {k}: {v:.3f}')
    

In [ ]:
if not CACHE_LOADED:
    # T1-T8: TEMPORAL TESTS (REVISED + T8 NEW)
    # T1: Spearman rho per subregion
    # T2: Near-duplicate rate (< 1%)
    # T3: Delta R2 + Directional AUC
    # T4: RANO criterion (ET +40%)
    # T5: Coherence dual-bound 0.70-0.93
    # T7: Cohen's d on distances (not norms)
    # T8: Kendall tau trajectory monotonicity (NEW)
    print('\n' + '='*60)
    print('  TEMPORAL TESTS T1-T8 (Static Modeling Limitations)')
    print('='*60)
    from scipy.stats import kendalltau as kt
    
    for mn, embs in models.items():
        keys = list(embs.keys())
        # L2-normalise embeddings before all temporal distance metrics
        raw  = np.stack([embs[k] for k in keys])
        norms_for_l2 = np.linalg.norm(raw, axis=1, keepdims=True) + 1e-8
        embs_l2 = {k: embs[k] / norms_for_l2[i] for i, k in enumerate(keys)}
        pe   = {}
        for k in keys:
            pid, tp = k.split('__')
            if pid not in pe: pe[pid] = {}
            pe[pid][tp] = embs[k]
        longi = {p: t for p, t in pe.items() if len(t) >= 2}
        print(f'  {mn}: {len(longi)} longitudinal patients')
        if len(longi) < 5:
            for t in ['T1','T2','T3','T4','T5','T6','T7','T8']:
                results[mn][t] = 0
            continue
    
        den, dvol_wt, dvol_et, csim = [], [], [], []
        v_first, v_last = [], []  # for T7
    
        for pid, tps in longi.items():
            stps = sorted(tps.keys())
            e_first = embs_l2['{}__{}'.format(pid, stps[0])]; e_last = embs_l2['{}__{}'.format(pid, stps[-1])]
            v_first.append(np.linalg.norm(e_last - e_first))  # T7
    
            for i in range(len(stps) - 1):
                e0 = embs_l2.get('{}__{}'.format(pid, stps[i]),   tps[stps[i]])
                e1 = embs_l2.get('{}__{}'.format(pid, stps[i+1]), tps[stps[i+1]])
                d = np.linalg.norm(e1 - e0)
                den.append(d)
                n0 = np.linalg.norm(e0) + 1e-8
                n1 = np.linalg.norm(e1) + 1e-8
                csim.append(float((e0/n0) @ (e1/n1)))
                if tumor_df is not None:
                    v0 = tumor_df[(tumor_df['patient_id']==pid) & (tumor_df['timepoint']==int(stps[i]))]
                    v1 = tumor_df[(tumor_df['patient_id']==pid) & (tumor_df['timepoint']==int(stps[i+1]))]
                    if len(v0) > 0 and len(v1) > 0:
                        dvol_wt.append(abs(v1.iloc[0]['wt_vol'] - v0.iloc[0]['wt_vol']))
                        et0 = v0.iloc[0]['et_vol']; et1 = v1.iloc[0]['et_vol']
                        dvol_et.append(et1 / (et0 + 1e-6) - 1)  # fractional ET change
    
        den = np.array(den)
    
        # T1 — Spearman rho per subregion
        ml = min(len(den), len(dvol_wt))
        if ml >= 5:
            rho_wt, _ = spearmanr(den[:ml], dvol_wt[:ml])
            results[mn]['T1_spearman_wt'] = abs(float(rho_wt))
        else:
            results[mn]['T1_spearman_wt'] = 0
    
        # T2 — Near-duplicate rate
        near_dup = np.mean(den < 0.001 * den.mean()) if len(den) > 0 else 1.0
        results[mn]['T2_ordering_pass'] = float(near_dup < 0.01)  # 1=pass, 0=fail
    
        # T3 — Delta R2 + Directional AUC
        ml_et = min(len(den), len(dvol_et))
        if ml_et >= 10:
            yd = np.array(dvol_wt[:ml_et])
            pd3 = cross_val_predict(Ridge(1.0), den[:ml_et].reshape(-1,1), yd, cv=min(5,ml_et//2))
            results[mn]['T3_delta_R2'] = float(r2_score(yd, pd3))  # can be negative
            vol_sign = (np.array(dvol_wt[:ml_et]) > 0).astype(int)
            drift_sign = (den[:ml_et] > np.median(den[:ml_et])).astype(int)
            if len(set(vol_sign)) > 1:
                from sklearn.metrics import roc_auc_score as ras
                results[mn]['T3_directional_auc'] = float(ras(vol_sign, drift_sign))
            else:
                results[mn]['T3_directional_auc'] = 0.5
        else:
            results[mn]['T3_delta_R2'] = 0
            results[mn]['T3_directional_auc'] = 0.5
    
        # T4 — RANO Response AUC (ET +40%)
        if ml_et >= 10 and tumor_df is not None:
            progressive = (np.array(dvol_et[:ml_et]) > 0.40).astype(int)
            if len(set(progressive)) > 1:
                from sklearn.metrics import roc_auc_score as ras
                results[mn]['T4_rano_auc'] = float(ras(progressive, den[:ml_et]))
            else:
                results[mn]['T4_rano_auc'] = 0.5
        else:
            results[mn]['T4_rano_auc'] = 0.5
    
        # T5 — Coherence dual-bound
        coherence = float(np.mean(csim)) if csim else 0
        results[mn]['T5_coherence'] = coherence
        results[mn]['T5_pass_dual'] = float(0.70 < coherence < 0.93)
    
        # T6 — Velocity CV (descriptor)
        results[mn]['T6_velocity_cv'] = float(np.std(den) / (np.mean(den) + 1e-8)) if len(den) else 0
    
        # T7 — Cohen's d on distances (progressors vs stable)
        all_dists = np.array(v_first)
        if tumor_df is not None and len(all_dists) >= 10:
            pids_longi = list(longi.keys())
            prog_mask = []
            for pid in pids_longi:
                stps = sorted(longi[pid].keys())
                v0 = tumor_df[(tumor_df['patient_id']==pid) & (tumor_df['timepoint']==int(stps[0]))]
                vT = tumor_df[(tumor_df['patient_id']==pid) & (tumor_df['timepoint']==int(stps[-1]))]
                if len(v0) > 0 and len(vT) > 0:
                    prog_mask.append((vT.iloc[0]['et_vol'] / (v0.iloc[0]['et_vol'] + 1e-6) - 1) > 0.40)
                else:
                    prog_mask.append(False)
            prog_mask = np.array(prog_mask)
            d_prog = all_dists[prog_mask]; d_stab = all_dists[~prog_mask]
            if len(d_prog) >= 3 and len(d_stab) >= 3:
                pooled = np.sqrt((np.var(d_prog) + np.var(d_stab)) / 2) + 1e-8
                results[mn]['T7_treatment_d'] = float(abs(d_prog.mean() - d_stab.mean()) / pooled)
            else:
                results[mn]['T7_treatment_d'] = 0
        else:
            results[mn]['T7_treatment_d'] = 0
    
        # T8 — Kendall tau trajectory monotonicity (NEW)
        taus = []
        for pid, tps in longi.items():
            stps = sorted(tps.keys())
            if len(stps) < 2: continue  # 2+ visits sufficient for Kendall tau
            dists = [np.linalg.norm(tps[v] - tps[stps[0]]) for v in stps[1:]]
            tau, _ = kt(dists, range(len(dists)))
            taus.append(tau)
        results[mn]['T8_kendall_tau'] = float(np.nanmean(taus)) if taus else 0
    
        for k, v in sorted(results[mn].items()):
            if k.startswith('T'):
                flag = ' <- WEAK (static CNN limitation)' if v < 0.30 and k in [
                    'T1_spearman_wt','T3_delta_R2','T3_directional_auc','T4_rano_auc','T8_kendall_tau'] else ''
                print(f'  {mn} {k}: {v:.3f}{flag}')
    

In [ ]:
# FULL 18-TEST DASHBOARD + SAVE RESULTS
print("\n" + "="*60)
print("  FULL 18-TEST DASHBOARD")
print("="*60)

THRESHOLDS = {
    "M1_volume_R2_ridge":     (0.50, "low-pri"),
    "M1_volume_R2_rf":        (0.50, "med-pri"),
    "M1_spearman_rho":        (0.55, "med-pri"),
    "M2_logvol_R2_ridge":     (0.40, "low-pri"),
    "M2_logvol_R2_rf":        (0.40, "med-pri"),
    "M3_enhancement_ridge":   (0.25, "low-pri"),
    "M3_enhancement_rf":      (0.25, "low-pri"),
    "M4_necrosis_F1":         (0.60, "med-pri"),
    "M5_corefrac_ridge":      (0.30, "low-pri"),
    "M5_corefrac_rf":         (0.30, "low-pri"),
    "M6_patient_purity_pct":  (60.0, "HIGH-PRI"),
    "H1_rankme":              (30.0,  "med-pri"),
    "H1_eff_rank_95":         (50.0,  "med-pri"),
    "H2_diversity":            (0.25, "med-pri"),
    "H2_uniformity":          (-3.0,  "med-pri"),
    "H3_responder_F1":         (0.55, "med-pri"),
    "H4_norm_cv":              (0.30, "low-pri"),
    "H5_rankme_standalone":   (30.0,  "med-pri"),
    "T1_spearman_wt":          (0.30, "HIGH-PRI"),
    "T2_ordering_pass":        (1.0,  "low-pri"),
    "T3_delta_R2":             None,
    "T3_directional_auc":      (0.55, "low-pri"),
    "T4_rano_auc":             (0.65, "HIGH-PRI"),
    "T5_coherence":            (0.70, "med-pri"),
    "T5_pass_dual":            (1.0,  "low-pri"),
    "T6_velocity_cv":          None,
    "T7_treatment_d":          (0.50, "HIGH-PRI"),
    "T8_kendall_tau":          (0.30, "HIGH-PRI"),
}
LOWER_BETTER = {"H4_norm_cv"}

for mn in models:
    r = results.get(mn, {})
    passed = total = 0
    high_pri_fails = []
    for k, tup in THRESHOLDS.items():
        if tup is None or k not in r: continue
        thresh, pri = tup
        v = r[k]
        total += 1
        is_nan = isinstance(v, float) and (v != v)  # nan check
        ok = False if is_nan else ((v <= thresh) if k in LOWER_BETTER else (v >= thresh))
        if ok: passed += 1
        if pri == "HIGH-PRI" and not ok:
            high_pri_fails.append((k, v, thresh, is_nan))
    print(f"\n{mn}  [{passed}/{total} pass]")
    for k, tup in THRESHOLDS.items():
        if tup is None:
            v = r.get(k)
            if v is not None: print(f"  {k:<35s} {v:>8.3f}  (descriptor)")
            continue
        if k not in r: continue
        thresh, pri = tup
        v = r[k]
        is_nan = isinstance(v, float) and (v != v)
        if is_nan:
            print(f"  {k:<35s} {'nan':>8s}  thresh={str(thresh):<6s}  FAIL (NaN-not computed)  [{pri}]")
            continue
        ok = (v <= thresh) if k in LOWER_BETTER else (v >= thresh)
        flag = "PASS" if ok else "FAIL"
        print(f"  {k:<35s} {v:>8.3f}  thresh={str(thresh):<6s}  {flag}  [{pri}]")
    if high_pri_fails:
        print("\n  HIGH-PRIORITY FAILS:")
        for k, v, t, is_nan in high_pri_fails:
            if is_nan:
                print(f"    {k} = NaN  (not computed — needs re-run without cache)")
            else:
                print(f"    {k} = {v:.3f} -> needs > {t:.2f}")

# ── Save JSON results ──
json_path = OUTPUT_ROOT / "cnn_eval_results.json"
serializable = {}
for mn, r in results.items():
    serializable[mn] = {k: (None if (isinstance(v, float) and v!=v) else
                             float(v) if isinstance(v, (np.floating, float)) else v)
                        for k, v in r.items()}
with open(json_path, "w") as f:
    _json.dump(serializable, f, indent=2)
print(f"\nSaved: {json_path}")

# ── t-SNE + distribution plots (only when embeddings are available) ──
if CACHE_LOADED:
    print("\n⚠ t-SNE skipped in cache mode (no embeddings loaded).")
    print("  To generate plots: attach cnn_nnunet_embeddings.npz and set SKIP_IF_CACHED=False")
else:
    print("\nGenerating t-SNE visualization...")
    for mn, emb_dict in models.items():
        if not emb_dict:
            print(f"  {mn}: no embeddings — skipping plot")
            continue
        keys = sorted(emb_dict.keys())
        X = np.stack([emb_dict[k] for k in keys])
        X_norm = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-8)
        pids = [k.split("__")[0] for k in keys]
        tps  = [int(k.split("__")[1]) for k in keys]

        X_pca50 = PCA(n_components=min(50, X_norm.shape[1])).fit_transform(X_norm)
        X_tsne = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(X_pca50)

        fig, axes = plt.subplots(1, 2, figsize=(16, 7))
        sc = axes[0].scatter(X_tsne[:, 0], X_tsne[:, 1], c=tps, cmap="viridis", s=8, alpha=0.6)
        plt.colorbar(sc, ax=axes[0], label="Timepoint")
        axes[0].set_title(f"{mn} — t-SNE colored by timepoint")
        axes[0].set_xlabel("t-SNE 1"); axes[0].set_ylabel("t-SNE 2")

        if tumor_df is not None:
            wt_col = next((c for c in tumor_df.columns if "wt" in c.lower() and "vol" in c.lower()), None)
            vols = []
            for k in keys:
                pid, tp = k.split("__")
                row = match_row(tumor_df, pid, tp)
                vols.append(float(row[wt_col]) if (row is not None and wt_col) else 0)
            sc2 = axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1],
                                  c=np.log1p(vols), cmap="hot", s=8, alpha=0.6)
            plt.colorbar(sc2, ax=axes[1], label="log(1 + WT volume)")
            axes[1].set_title(f"{mn} — t-SNE colored by tumor volume")
        else:
            axes[1].text(0.5, 0.5, "No volume data", ha="center", transform=axes[1].transAxes)
        axes[1].set_xlabel("t-SNE 1"); axes[1].set_ylabel("t-SNE 2")

        plt.tight_layout()
        out = FIG_DIR / f"{mn}_tsne.png"
        plt.savefig(out, dpi=150, bbox_inches="tight"); plt.close()
        print(f"  Saved: {out}")

    # ── Embedding norm + PCA distribution ──
    for mn, emb_dict in models.items():
        if not emb_dict: continue
        keys = sorted(emb_dict.keys())
        X = np.stack([emb_dict[k] for k in keys])
        norms = np.linalg.norm(X, axis=1)
        X_norm = X / (norms[:, None] + 1e-8)

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        axes[0].hist(norms, bins=50, color="steelblue", alpha=0.8)
        axes[0].axvline(norms.mean(), color="red", ls="--", label=f"mean={norms.mean():.2f}")
        axes[0].set_title(f"{mn} — Embedding L2 norms"); axes[0].legend()

        pca = PCA(n_components=min(100, X_norm.shape[1])).fit(X_norm)
        axes[1].plot(np.cumsum(pca.explained_variance_ratio_), color="darkorange", lw=2)
        axes[1].axhline(0.95, color="gray", ls="--", alpha=0.5, label="95%")
        axes[1].set_title(f"{mn} — PCA cumulative variance"); axes[1].legend()

        svs = np.linalg.svd(X_norm[:min(500, len(X_norm))], compute_uv=False)
        axes[2].plot(svs[:100], color="green", lw=2)
        axes[2].set_title(f"{mn} — Singular value spectrum")

        plt.tight_layout()
        out = FIG_DIR / f"{mn}_distributions.png"
        plt.savefig(out, dpi=150, bbox_inches="tight"); plt.close()
        print(f"  Saved: {out}")

print(f"\nAll outputs in: {OUTPUT_ROOT}")


In [ ]:
# CNN PHASE 2 — SUMMARY (computed from results, NOT hardcoded)
print("\n" + "="*60)
print("  PHASE 2 CNN EVALUATION — COMPUTED SUMMARY")
print("="*60)

for mn in models:
    r = results.get(mn, {})
    print(f"\nModel: {mn}")
    # Use known values (works in both live and cached mode)
    emb_vals = list(models[mn].values())
    dim = len(emb_vals[0]) if emb_vals else 2825  # CNN: 2825-D
    n_scans = len(models[mn]) if models[mn] else results[mn].get("_n_scans", 1620)
    n_pats  = len(set(k.split("__")[0] for k in models[mn].keys())) if models[mn] else results[mn].get("_n_patients", 731)
    print(f"  Scans: {n_scans} | Embedding dim: {dim}")

    print(f"\n  MORPHOLOGY:")
    for k in ["M1_spearman_rho", "M1_volume_R2_rf", "M2_logvol_R2_rf",
              "M4_necrosis_F1", "M6_patient_purity_pct"]:
        v = r.get(k)
        if v is not None:
            print(f"    {k:<35s} = {v:.3f}")

    print(f"\n  HETEROGENEITY:")
    for k in ["H1_rankme", "H2_diversity", "H3_responder_F1", "H4_norm_cv"]:
        v = r.get(k)
        if v is not None:
            fmt = ".1f" if "rankme" in k else ".3f"
            print(f"    {k:<35s} = {v:{fmt}}")

    print(f"\n  TEMPORAL:")
    for k in ["T1_spearman_wt", "T4_rano_auc", "T7_treatment_d", "T8_kendall_tau"]:
        v = r.get(k)
        if v is not None:
            print(f"    {k:<35s} = {v:.3f}")

    # Strengths & weaknesses (computed)
    strengths = []
    weaknesses = []
    THRESH2 = {
        "M1_spearman_rho": 0.55, "M4_necrosis_F1": 0.60, "H1_rankme": 30.0,
        "H2_diversity": 0.25, "T5_coherence": 0.70, "H3_responder_F1": 0.55,
        "M6_patient_purity_pct": 60.0, "T4_rano_auc": 0.65,
        "T7_treatment_d": 0.50, "T8_kendall_tau": 0.30
    }
    for k, thresh in THRESH2.items():
        v = r.get(k)
        if v is None: continue
        if v >= thresh:
            strengths.append(f"{k} = {v:.3f} (>{thresh})")
        else:
            weaknesses.append(f"{k} = {v:.3f} (needs >{thresh})")

    print(f"\n  STRENGTHS ({len(strengths)}):")
    for s in strengths:
        print(f"    ✅ {s}")

    print(f"\n  LIMITATIONS ({len(weaknesses)}) → Phase 3 targets:")
    for w in weaknesses:
        print(f"    ❌ {w}")

print("\n" + "="*60)
print("  PHASE 2 COMPLETE")
print("="*60)
print(f"  Results saved: {OUTPUT_ROOT / 'cnn_eval_results.json'}")
print(f"  Figures saved: {FIG_DIR}")
print(f"  → Phase 3: SwinUNETR + temporal sequences")
